In [ ]:
# %% imports and environment setup
from importlib import import_module
from pathlib import Path
import sys, os, multiprocessing as mp
import imageio_ffmpeg
import torch
from torch._functorch._aot_autograd.logging_utils import model_name

# 🧠 use all logical cores efficiently
LOGICAL = mp.cpu_count() or 8
os.environ["OMP_NUM_THREADS"] = str(LOGICAL)
os.environ["MKL_NUM_THREADS"] = str(LOGICAL)
os.environ["OPENBLAS_NUM_THREADS"] = str(LOGICAL)
os.environ["NUMEXPR_NUM_THREADS"] = str(LOGICAL)

torch.set_num_threads(LOGICAL // 2)
torch.set_num_interop_threads(LOGICAL // 2)

# ✅ detect Intel oneDNN (IPEX) if available
try:
    import intel_extension_for_pytorch as ipex
    HAS_IPEX = True
    print("[info] Intel Extension for PyTorch (IPEX) detected.")
except ImportError:
    HAS_IPEX = False
    print("[info] IPEX not found, running standard PyTorch.")

# add repo root
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# import internal tools
input_selector = import_module("tools.input_selector")
ffmpeg_tools = import_module("tools.FFmpeg.FFmpeg_utilization")
model_selector = import_module("tools.model_selector")
report = import_module("tools.preview_report")

# check hardware
print(f"[cpu] threads: {LOGICAL}")
print(f"[gpu] available: {torch.cuda.is_available()}")


In [ ]:
# %% [markdown]
# ## 2️⃣ Input: Select video
# Either type a path manually or assign one directly for faster testing.

# %%
VIDEO_EXTS = {".mp4", ".mov", ".mkv", ".avi", ".webm", ".m4v"}

input_selector = import_module("tools.input_selector")
video_path = input_selector.get_input_video_path(allowed_exts=VIDEO_EXTS)

# Example: hard-code during testing
# video_path = ROOT / "demoGreyscale.mp4"

print(f"[ok] selected video: {video_path}")


In [ ]:
# %% [markdown]
# ## 3️⃣ Extract frames using FFmpeg
# Uses your tools/FFmpeg/FFmpeg_utilization.py

# %%
temp_root = ROOT / "temp"
temp_root.mkdir(parents=True, exist_ok=True)

ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()
ffmpeg_tools = import_module("tools.FFmpeg.FFmpeg_utilization")

frames_dir = ffmpeg_tools.extract_frames(ffmpeg_path, video_path, temp_root)

if not frames_dir or not Path(frames_dir).exists():
    raise RuntimeError("Frame extraction failed.")
else:
    print(f"[ok] extracted frames dir: {frames_dir}")


In [ ]:
# %% 4️⃣ prepare output + run model
frames_path = Path(frames_dir)
color_dir = frames_path.parent / f"{frames_path.name}_colorized"
color_dir.mkdir(parents=True, exist_ok=True)

use_gpu = torch.cuda.is_available()

# optional variant (only used by Zhang)
try:
    zhang_variant
except NameError:
    zhang_variant = None

if HAS_IPEX and not use_gpu:
    print("[info] enabling Intel oneDNN optimizations...")
    ipex.enable_onednn_fusion(True)
    try:
        ipex.set_fp32_math_mode(mode="BF16")
    except Exception:
        pass

# auto batch sizing
batch_size = max(1, LOGICAL // 2)

print(f"[start] running model: {model_name} (batch={batch_size})")
model_selector.run_colorizer(
    model_name="colorize_zhang",
    frames_dir=frames_path,
    color_dir=color_dir,
    models_dir=ROOT / "models",
    zhang_variant=zhang_variant,
    preview=False,
    use_gpu=use_gpu,
    batch_size=batch_size,
    num_threads=LOGICAL,
    input_size=224,
    progress=True,
    prefetch_workers=LOGICAL // 4,
    save_workers=2,
)
print(f"[ok] colorization complete: {color_dir}")


In [4]:
# %% 5️⃣ Temporal Smoothing Step
from pathlib import Path
from tools.TemporalSmoothing import apply_temporal_smoothing

def _normalize_path(p: str | Path) -> Path | None:
    """Clean user path input (handles Windows backslashes)."""
    if not p:
        return None
    p = str(p).strip().strip('"').strip("'").replace("\\", "/")
    return Path(p).expanduser().resolve()

# ------------------ ask for input ------------------
try:
    color_dir
except NameError:
    color_dir = None

raw_in = input("Enter input folder for smoothing (blank = last colorized output): ").strip()
input_folder = _normalize_path(raw_in) if raw_in else color_dir

if not input_folder or not Path(input_folder).exists():
    print("[error] No valid input folder found. Did you run the colorization cell first?")
    raise SystemExit

# ------------------ auto output ------------------
smooth_dir = input_folder.parent / f"{input_folder.name}_TemporalSmoothed"
smooth_dir.mkdir(parents=True, exist_ok=True)

# ------------------ smoothing window ------------------
try:
    window_size = int(input("Enter temporal smoothing window (odd number, default=9): ") or 9)
    if window_size % 2 == 0:
        window_size -= 1
    if window_size < 3:
        window_size = 3
except Exception:
    window_size = 9

print(f"[info] smoothing {input_folder} → {smooth_dir} (window={window_size})")

# ------------------ run ------------------
apply_temporal_smoothing(
    input_folder=input_folder,
    output_folder=smooth_dir,
    use_onnx=True,
    window_size=window_size,
)

print(f"[ok] temporal smoothing complete: {smooth_dir}")


[info] smoothing C:\Users\bebo\Documents\GitHub\VideoRasterization\temp\20251016_235419\frames_colorized → C:\Users\bebo\Documents\GitHub\VideoRasterization\temp\20251016_235419\frames_colorized_TemporalSmoothed (window=9)
[warn] onnxruntime not available. Using NumPy averaging.
[warn] OpenCV unavailable; skipping smoothing and copying frames verbatim.
[detail] OpenCV import error: Error importing numpy: you should not try to import numpy from
        its source directory; please exit the numpy source tree, and relaunch
        your python interpreter from there.
[done] Temporal smoothing skipped (OpenCV dependency missing).
[ok] temporal smoothing complete: C:\Users\bebo\Documents\GitHub\VideoRasterization\temp\20251016_235419\frames_colorized_TemporalSmoothed


In [ ]:
# %% [markdown]
# ## 6️⃣ (Optional) Rebuild video or generate report
# You can add temporal smoothing or report steps later here.

# %%
# example placeholders:
# ffmpeg_tools.rebuild_video(...)
# report = import_module("tools.preview_report")
# report.generate_report(...)

print("[done] pipeline finished (frames ready).")